# Federated LSTM for OpenSky FL-IDS -- Pretrial Notebook

Trains a small **LSTM-based intrusion detector** across simulated regional
"clients" using **FedAvg**, on top of the labeled OpenSky data produced by
`opensky_fl_ids_pipeline.py`.

**Why LSTM here (vs. the earlier feed-forward `federated_simulation.py`):**
each aircraft's stream of state-vector messages is a *sequence* -- an
attack often shows up as a pattern over several consecutive messages
(e.g. a sudden implausible jump), not just a single row in isolation. The
LSTM looks at a short window of an aircraft's recent messages at once,
instead of judging each row independently.

**How each "client" is defined:** same as before -- each `region_cell`
(coarse lat/lon grid cell) stands in for a ground receiver / receiver
cluster. Each client only ever trains on its own local sequences; only
model weights are shared and averaged centrally (FedAvg), never raw data.

**Runs on Google Colab or Kaggle.** Upload your labeled CSV (the output
of `opensky_fl_ids_pipeline.py`) using the cell below, or edit `DATA_PATH`
to point at it directly (e.g. a Kaggle dataset path under `/kaggle/input/...`).

> This is a **pretrial / prototype** -- a first pass to check the FL+LSTM
> mechanics work and get a rough read on detectability, not a
> production-grade or fully tuned model.


## 1. Setup

In [ ]:
# Colab and Kaggle both ship torch, pandas, scikit-learn, numpy, matplotlib
# by default -- this just makes sure they're present without reinstalling
# if they already are. The `|| true` lets this cell continue harmlessly
# if pip is externally-managed and blocks reinstall (common on some
# Kaggle images) -- in that case the packages are already there anyway.
!pip install -q torch pandas scikit-learn numpy matplotlib || true


In [ ]:
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

RNG_SEED = 42
torch.manual_seed(RNG_SEED)
np.random.seed(RNG_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


## 2. Load your labeled data

**Option A (Colab):** run the upload cell below and pick your
`labeled_*.csv` / `combined_labeled.csv` file.

**Option B (Kaggle, or if you already have the file on disk / in a Kaggle
dataset):** skip the upload cell and just set `DATA_PATH` directly, e.g.
`DATA_PATH = "/kaggle/input/opensky-labeled/combined_labeled.csv"`.


In [ ]:
# --- Option A: Colab upload (skip this cell on Kaggle) ---
DATA_PATH = None
try:
    from google.colab import files
    uploaded = files.upload()
    DATA_PATH = list(uploaded.keys())[0]
    print("Uploaded:", DATA_PATH)
except ImportError:
    print("Not running in Colab -- skip this cell and set DATA_PATH manually below.")


In [ ]:
# --- Option B: set this manually if not using the Colab upload above ---
if DATA_PATH is None:
    DATA_PATH = "combined_labeled.csv"  # <-- EDIT THIS on Kaggle / local

print("Loading:", DATA_PATH)
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"{len(df):,} rows, {df['icao24'].nunique():,} aircraft, "
      f"{df['region_cell'].nunique():,} region cells")
df["attack_type"].value_counts()


## 3. Config

The main knobs you'll want to tune for your actual dataset size:

- `SEQ_LEN` / `STRIDE`: how many consecutive messages the LSTM looks at
  per prediction, and how much consecutive windows overlap.
- `MIN_CLIENT_SEQS`: clients (regions) with fewer sequences than this are
  dropped -- too little local data to train anything meaningful. Raise
  this once you're running on a full day+ of data; it's set low here so
  a single-hour test file still produces usable clients.
- `N_ROUNDS` / `LOCAL_EPOCHS`: FedAvg communication rounds and local
  training epochs per client per round.
- `USE_SQRT_CLASS_WEIGHTS`: on a benign-heavy imbalanced dataset with
  small per-client sample sizes, full "balanced" class weighting can make
  FedAvg unstable (verified while building this notebook -- it dropped
  accuracy to ~10% and wouldn't recover). Square-root-dampened weights
  converged cleanly to ~90%+ in that same test. Leave this on unless
  you've verified full weighting is stable for your data size.


In [ ]:
SEQ_LEN = 10
STRIDE = 5
MIN_CLIENT_SEQS = 200        # raise this for full-day+ datasets
N_ROUNDS = 15
LOCAL_EPOCHS = 3
CLIENT_FRACTION = 1.0        # fraction of clients sampled each round
BATCH_SIZE = 128
LR = 1e-3
HIDDEN_SIZE = 32
USE_SQRT_CLASS_WEIGHTS = True

FEATURE_COLS = [
    "implied_speed_mps", "speed_consistency_delta", "vrate_consistency_delta",
    "inter_arrival_s", "turn_rate_deg_s", "track_message_count",
    "is_physically_implausible", "squawk_changed", "icao24_valid_format",
    "is_stale", "region_avg_inter_arrival", "is_low_coverage_region",
    "is_climbing_or_descending",
]
BOOL_COLS = ["is_physically_implausible", "squawk_changed", "icao24_valid_format",
             "is_stale", "is_low_coverage_region", "is_climbing_or_descending"]

missing = [c for c in FEATURE_COLS + ["attack_type", "region_cell", "icao24", "time"] if c not in df.columns]
assert not missing, f"Missing expected columns from opensky_fl_ids_pipeline.py output: {missing}"


## 4. Build per-aircraft sequences

For each aircraft (`icao24`) within each region, this slides a
`SEQ_LEN`-message window (stepping by `STRIDE`) over its messages sorted
by time, and labels each window with the `attack_type` of its *last*
message -- i.e. "given the last `SEQ_LEN` messages, is this aircraft's
current state benign or under attack, and what kind?"


In [ ]:
def build_sequences(df):
    d = df.copy()
    for c in BOOL_COLS:
        d[c] = d[c].astype(float)
    d[FEATURE_COLS] = d[FEATURE_COLS].fillna(-1)

    seqs, labels, regions = [], [], []
    for (icao, region), g in d.groupby(["icao24", "region_cell"], sort=False):
        g = g.sort_values("time")
        if len(g) < SEQ_LEN:
            continue
        feats = g[FEATURE_COLS].values.astype(np.float32)
        labs = g["attack_type"].values
        for start in range(0, len(g) - SEQ_LEN + 1, STRIDE):
            seqs.append(feats[start:start + SEQ_LEN])
            labels.append(labs[start + SEQ_LEN - 1])
            regions.append(region)
    return np.stack(seqs), np.array(labels), np.array(regions)

print("Building sequences (this is the slowest step -- one pass per aircraft track)...")
X, y_raw, regions = build_sequences(df)
print(f"{len(X):,} sequences built, shape {X.shape}  (n_sequences, SEQ_LEN, n_features)")

le = LabelEncoder()
y = le.fit_transform(y_raw)
class_names = le.classes_
print("Classes:", list(class_names))


In [ ]:
# Scale features (fit on all data before partitioning into clients --
# same simplification noted in federated_simulation.py: a real deployment
# would fit on a public reference sample or use a federated-safe scaling
# scheme instead of pooled data).
n_samples, seq_len, n_feat = X.shape
scaler = StandardScaler()
X_flat = scaler.fit_transform(X.reshape(-1, n_feat))
X = X_flat.reshape(n_samples, seq_len, n_feat)


## 5. Partition into FL clients by region

In [ ]:
clients = {}
dropped = 0
for region in np.unique(regions):
    idx = np.where(regions == region)[0]
    if len(idx) < MIN_CLIENT_SEQS:
        dropped += 1
        continue
    rng = np.random.default_rng(RNG_SEED)
    shuffled = rng.permutation(idx)
    split = int(len(shuffled) * 0.8)
    train_idx, test_idx = shuffled[:split], shuffled[split:]
    clients[region] = dict(
        X_train=X[train_idx], y_train=y[train_idx],
        X_test=X[test_idx], y_test=y[test_idx],
    )

print(f"{len(clients)} clients built (min {MIN_CLIENT_SEQS:,} sequences each), "
      f"{dropped} region(s) dropped for insufficient data")
assert len(clients) >= 2, (
    "Fewer than 2 usable clients -- lower MIN_CLIENT_SEQS or SEQ_LEN, "
    "or provide more data (more hours/days)."
)

X_global_test = np.concatenate([c["X_test"] for c in clients.values()])
y_global_test = np.concatenate([c["y_test"] for c in clients.values()])


## 6. LSTM model

Small single-layer LSTM -- kept lightweight since (as with the
feed-forward version) FedAvg's communication cost scales with model size,
and this is a pretrial, not a final architecture.


In [ ]:
class LSTMIDSNet(nn.Module):
    def __init__(self, n_features, hidden_size, n_classes):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, n_classes)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return self.fc(h_n[-1])


def model_size_bytes(model):
    return sum(p.numel() * p.element_size() for p in model.parameters())


## 7. FedAvg training loop

Same pattern as `federated_simulation.py`: each round, sampled clients
train locally on their own sequences only, send weights (never data) to
be averaged, then receive the new global weights back.


In [ ]:
class_counts = np.bincount(y, minlength=len(class_names))
weights_np = len(y) / (len(class_names) * class_counts)
if USE_SQRT_CLASS_WEIGHTS:
    weights_np = np.sqrt(weights_np)
class_weights = torch.tensor(weights_np, dtype=torch.float32).to(DEVICE)
print("Class weights:", dict(zip(class_names, weights_np.round(2))))

global_model = LSTMIDSNet(n_feat, HIDDEN_SIZE, len(class_names)).to(DEVICE)
payload_bytes = model_size_bytes(global_model)
print(f"Model size: {payload_bytes / 1024:.1f} KB per client, per round")

def local_train(model, X_train, y_train):
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train, dtype=torch.long))
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True)
    for _ in range(LOCAL_EPOCHS):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            opt.step()
    return model.state_dict()

def federated_average(state_dicts, sample_weights):
    total = sum(sample_weights)
    avg = copy.deepcopy(state_dicts[0])
    for key in avg:
        avg[key] = sum(sd[key].float() * (w / total) for sd, w in zip(state_dicts, sample_weights))
    return avg

@torch.no_grad()
def evaluate(model, X_test, y_test):
    model.eval()
    xb = torch.tensor(X_test).to(DEVICE)
    preds = model(xb).argmax(dim=1).cpu().numpy()
    return preds, (preds == y_test).mean()


In [ ]:
client_names = list(clients.keys())
rng = np.random.default_rng(RNG_SEED)
total_bytes_transferred = 0
round_accuracies = []
comm_cumulative = []

print(f"Running FedAvg: {N_ROUNDS} rounds, {LOCAL_EPOCHS} local epochs/round, "
      f"{len(clients)} clients ({CLIENT_FRACTION:.0%} participation)\n")

for rnd in range(1, N_ROUNDS + 1):
    n_participants = max(1, int(len(client_names) * CLIENT_FRACTION))
    participants = rng.choice(client_names, size=n_participants, replace=False)

    local_states, sample_weights = [], []
    for cname in participants:
        local_model = copy.deepcopy(global_model).to(DEVICE)
        c = clients[cname]
        state = local_train(local_model, c["X_train"], c["y_train"])
        local_states.append(state)
        sample_weights.append(len(c["X_train"]))
        total_bytes_transferred += 2 * payload_bytes  # up + down

    global_model.load_state_dict(federated_average(local_states, sample_weights))

    _, acc = evaluate(global_model, X_global_test, y_global_test)
    round_accuracies.append(acc)
    comm_cumulative.append(total_bytes_transferred / 1e6)
    print(f"  Round {rnd:2d}/{N_ROUNDS} -- global test accuracy: {acc:.4f} "
          f"-- cumulative comm: {comm_cumulative[-1]:.2f} MB")


## 8. Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(range(1, N_ROUNDS + 1), round_accuracies, marker="o")
axes[0].set_xlabel("Round"); axes[0].set_ylabel("Global test accuracy")
axes[0].set_title("Accuracy over FedAvg rounds"); axes[0].grid(alpha=0.3)

axes[1].plot(range(1, N_ROUNDS + 1), comm_cumulative, marker="o", color="darkorange")
axes[1].set_xlabel("Round"); axes[1].set_ylabel("Cumulative MB transferred")
axes[1].set_title("Communication cost"); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()


**Note on interpreting accuracy:** this dataset is heavily imbalanced
(benign traffic dominates), so raw accuracy alone can be misleading --
predicting "benign" for everything would already score high. The
per-class report below (precision/recall/F1) is the more honest read on
whether each attack type is actually being detected.

In [ ]:
preds, _ = evaluate(global_model, X_global_test, y_global_test)
print(classification_report(y_global_test, preds, target_names=class_names, zero_division=0))


In [ ]:
print("=== Per-client local test accuracy (final global model) ===")
for cname, c in clients.items():
    _, acc = evaluate(global_model, c["X_test"], c["y_test"])
    print(f"  region {str(cname):<12s} n_train={len(c['X_train']):>7,}  local_test_acc={acc:.4f}")


## Next steps for this pretrial

- If accuracy on rarer attack types is weak, try: more `LOCAL_EPOCHS`,
  a larger `HIDDEN_SIZE`, a stacked/bidirectional LSTM, or more real data
  (this notebook was validated on a single hour -- multi-day data will
  behave very differently).
- Try `CLIENT_FRACTION < 1.0` to simulate clients dropping in/out, closer
  to real bandwidth-constrained FL.
- Compare against the feed-forward baseline in `federated_simulation.py`
  to see whether the sequence context actually helps for your data.
